## Importing Libraries

In [49]:
from ollama import chat
from ollama import ChatResponse
from ollama import embed
import json
import glob
from tqdm import tqdm
import os
import random

## Setting up files

In [50]:
GENERATION_MODEL = "qwen3:1.7b"
FILES = glob.glob("../Test_Files/clinical-diary*.txt")
PROMPT_FILE = "./prompts/parameter-extraction_prompt.txt"
OUTPUT_DIR = "./llm-outputs/parameter-extraction/"
OUTPUT_FILE = "experiment"

print(f"Found the following files {FILES}")

Found the following files ['../Test_Files\\clinical-diary_e1.txt']


## Pre-processing

Removal of unnecessary things from the file

In [51]:
pass

## Parameter Extraction

In this phase the parameters enforced by our client will be extracted from the unstructured clinical diary through a LLM approach

In [52]:
## Setting evironment
with open(PROMPT_FILE,"r", encoding="utf-8") as p:
    prompt_arr = [t.strip() for t in p.readlines() if t.strip()]
    base_prompt = " ".join(prompt_arr)

os.makedirs(OUTPUT_DIR,exist_ok=True)

count = 0

for path in os.listdir(OUTPUT_DIR):
    if os.path.isfile(os.path.join(OUTPUT_DIR, path)):
        count += 1

In [53]:
pbar = tqdm(total=len(FILES), desc="Processing diaries")

for file in FILES:
    with open(file,"r", encoding="utf-8") as f:
        text_arr = [t.strip() for t in f.readlines() if t.strip()]
        text = " ".join(text_arr)
        
        print(f"processing file: {file}")
        
        prompt = base_prompt.replace("{{DIARY_TEXT}}",text)
        
        stream = chat(
            model=GENERATION_MODEL,
            messages=[{"role": "user", "content": prompt}],
            stream=True,
            )
        
        llm_output = ""
        for chunk in stream:
            llm_output += chunk["message"]["content"]

        with open(f"{OUTPUT_DIR}{OUTPUT_FILE}-{count}.txt","w",encoding="utf-8") as o:
            o.write(f"Ouput for file {file}\n")
            o.write(f"{llm_output}\n\n")
            print(f"Saved LLM output on {OUTPUT_FILE}-{count}")
            
        
        print("\n")
        
        pbar.update(1)
        
pbar.close()

Processing diaries:   0%|          | 0/1 [00:00<?, ?it/s]

processing file: ../Test_Files\clinical-diary_e1.txt


Processing diaries: 100%|██████████| 1/1 [00:59<00:00, 59.24s/it]

Saved LLM output on experiment-0


